# Mi Código Projecto Tópicos Avanzados de Manipuladores Robóticos

#### Código para que MuJoCo se abra en pantalla completa y sobre los otros programas:

In [1]:
import ctypes
from ctypes import wintypes
import time

def traer_mujoco_al_frente(timeout=3.0):
    user32 = ctypes.WinDLL("user32", use_last_error=True)
    kernel32 = ctypes.WinDLL("kernel32", use_last_error=True)

    # Tipos explícitos para Python 64-bit
    WNDENUMPROC = ctypes.WINFUNCTYPE(wintypes.BOOL, wintypes.HWND, wintypes.LPARAM)
    user32.EnumWindows.argtypes = [WNDENUMPROC, wintypes.LPARAM]
    user32.GetWindowTextLengthW.argtypes = [wintypes.HWND]
    user32.GetWindowTextW.argtypes = [wintypes.HWND, wintypes.LPWSTR, ctypes.c_int]
    user32.IsWindowVisible.argtypes = [wintypes.HWND]
    user32.ShowWindow.argtypes = [wintypes.HWND, ctypes.c_int]
    user32.SetForegroundWindow.argtypes = [wintypes.HWND]
    user32.BringWindowToTop.argtypes = [wintypes.HWND]
    user32.GetForegroundWindow.restype = wintypes.HWND
    user32.GetWindowThreadProcessId.argtypes = [wintypes.HWND, ctypes.POINTER(wintypes.DWORD)]
    user32.GetWindowThreadProcessId.restype = wintypes.DWORD
    user32.AttachThreadInput.argtypes = [wintypes.DWORD, wintypes.DWORD, wintypes.BOOL]
    user32.keybd_event.argtypes = [wintypes.BYTE, wintypes.BYTE, wintypes.DWORD, ctypes.c_size_t]
    kernel32.GetCurrentThreadId.restype = wintypes.DWORD

    hwnd_mujoco = None
    inicio = time.time()

    # 1. Esperar hasta que el hilo del visor termine de crear la ventana
    while time.time() - inicio < timeout and hwnd_mujoco is None:
        def callback(hwnd, _):
            nonlocal hwnd_mujoco
            if user32.IsWindowVisible(hwnd):
                length = user32.GetWindowTextLengthW(hwnd)
                if length > 0:
                    buff = ctypes.create_unicode_buffer(length + 1)
                    user32.GetWindowTextW(hwnd, buff, length + 1)
                    if "MuJoCo" in buff.value:
                        hwnd_mujoco = hwnd
                        return False
            return True

        user32.EnumWindows(WNDENUMPROC(callback), 0)
        if hwnd_mujoco is None:
            time.sleep(0.05)

    if not hwnd_mujoco:
        return

    # 2. Enlazar la cola de entrada con la ventana activa (VS Code)
    hwnd_activo = user32.GetForegroundWindow()
    hilo_activo = user32.GetWindowThreadProcessId(hwnd_activo, None)
    hilo_python = kernel32.GetCurrentThreadId()

    if hilo_activo != hilo_python:
        user32.AttachThreadInput(hilo_python, hilo_activo, True)

    # 3. Simular toque de tecla Alt (0x12) para habilitar SetForegroundWindow en Windows
    user32.keybd_event(0x12, 0, 0, 0)
    user32.keybd_event(0x12, 0, 2, 0)

    # 4. Maximizar y traer al frente sobre VS Code
    user32.ShowWindow(hwnd_mujoco, 3)  # 3 = SW_MAXIMIZE
    user32.BringWindowToTop(hwnd_mujoco)
    user32.SetForegroundWindow(hwnd_mujoco)

    if hilo_activo != hilo_python:
        user32.AttachThreadInput(hilo_python, hilo_activo, False)

## Código Inicial: abrir Nero en su escena

In [35]:
import os
import sys
from pathlib import Path
import numpy as np
import mujoco
import mujoco.viewer
import time

# Primero, se fija el directorio de trabajo al directorio donde reside este script
if '__file__' in globals():
    BASE_DIR = Path(__file__).resolve().parent
else:
    # Soporte para Jupyter Notebook / IPython
    BASE_DIR = Path.cwd().resolve()

os.chdir(BASE_DIR)

xml_nero_filename = "scene_nero_arm.xml"
xml_cinta_filename = "cinta.xml"

# --- PARÁMETROS CONFIGURABLES DE LA CINTA ---
POSICION_CINTA = [-0.45, 0.0, 0.0]     # [X, Y, Z] en metros respecto al origen del mundo
ROTACION_CINTA = [0.0, 0.0, 1.57]   # [Roll, Pitch, Yaw], en radianes respecto al origen del mundo (90 rad = 90°)

# 1. Cargar ambas especificaciones XML en memoria
spec_escena = mujoco.MjSpec.from_file(xml_nero_filename)
spec_cinta = mujoco.MjSpec.from_file(xml_cinta_filename)

# 2. Crear un frame de anclaje en el worldbody y adjuntar la cinta
frame_cinta = spec_escena.worldbody.add_frame(
    pos=POSICION_CINTA,
    euler=ROTACION_CINTA
)
spec_escena.attach(spec_cinta, prefix="cinta_", frame=frame_cinta)

# 3. Compilar la escena unificada
model = spec_escena.compile()
data = mujoco.MjData(model)



# Código para establecer los valores inciales de posición de las articulaciones del robot

# Inicializamos el brazo en una posición no singular y en reposo, pero orientada a la cinta.
init_q = [0.0, 0.574, 0.0, 2.06, 0.0, 0.0, -0.927, 0.0423] # el vector tiene 8 valores, porque el brazo tiene 7 articulaciones y un gripper.
init_q_vel = [0.0] * 8

# Cargamos la posición y velocidad inicial al los valores del modelo, para que el brazo comience en el estado deseado.
data.qpos[:8] = init_q 
data.qvel[:8] = init_q_vel

#Tambien inicializamos la variable de control del brazo, para que la posición deseada inicial, sea igual a la posición inicial del robot y este no se mueva al comenzar la simulación
data.ctrl[:8] = init_q

# Podemos utilizar el simulador, dandolo un paso de tiempo, para que el brazo se ubique en la posición inicial deseada y llamar a su estado
mujoco.mj_step(model, data)

# Abre el viewer
with mujoco.viewer.launch_passive(model, data) as viewer:
    traer_mujoco_al_frente()
    
    # Configurar el estado inicial de la cámara libre
    viewer.cam.lookat[:] = [-0.1, 0, 0.2]  # Punto en el espacio al que mira la cámara [X, Y, Z]
    viewer.cam.distance = 1.6           # Distancia desde la cámara hasta el punto lookat
    viewer.cam.azimuth = 290             # Rotación horizontal en grados
    viewer.cam.elevation = -20          # Inclinación vertical en grados (negativo mira hacia abajo)

    # Configuramos la escena para evitar que la simulación corra lento    
    with viewer.lock():
        viewer.user_scn.flags[mujoco.mjtRndFlag.mjRND_SHADOW]=0
        viewer.user_scn.flags[mujoco.mjtRndFlag.mjRND_REFLECTION]=0
        viewer.user_scn.flags[mujoco.mjtRndFlag.mjRND_HAZE]=1
        viewer.user_scn.flags[mujoco.mjtRndFlag.mjRND_CULL_FACE]=1
        viewer.user_scn.flags[mujoco.mjtRndFlag.mjRND_SKYBOX]=1

    # Obtenemos el tiempo inicial
    start = time.time()
    
    # Corremos la simulación
    while viewer.is_running():

        # Actualizar la escena
        step_start = time.time()
        mujoco.mj_step(model,data)
        viewer.sync()
        
        # Actualizar el tiempo de simulación
        time_until_next = model.opt.timestep - (time.time() - step_start)
        if time_until_next > 0:
            time.sleep(time_until_next)